In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For better visuals
sns.set(style="whitegrid")

#  Load Dataset
df = pd.read_csv('RetailSales.csv')

df.head()

In [ ]:
# Shape of dataset
print("Shape of dataset:", df.shape)

# Column names
print("Columns:", df.columns)

# Data types
df.info()

# Check missing values
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract useful features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df.head()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Total_Sales'], bins=30, kde=True)
plt.title("Distribution of Total Sales")
plt.show()

# **Distribution of Total Sales :**

**Insight**: The distribution of 'Total Sales' is right-skewed, indicating that the majority of transactions have lower sales values, while a smaller number of transactions contribute to very high sales figures. This suggests that while there are many small purchases, a few large purchases significantly impact overall revenue.

# **STEP 2: Advanced EDA**

In [ ]:
# Total sales per category
category_sales = df.groupby('Category')['Total_Sales'].sum().sort_values(ascending=False)

print(category_sales)

# Visualization
plt.figure(figsize=(8,5))
sns.barplot(x=category_sales.index, y=category_sales.values)
plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.show()

# **Total Sales by Category:**
**Insight**: Contrary to the initial thought, the 'Clothing' category actually contributes the highest total sales, followed closely by 'Groceries'. 'Electronics' ranks third, indicating that focusing marketing or inventory efforts on Clothing and Groceries could yield the best returns.

In [ ]:
monthly_sales = df.groupby('Month')['Total_Sales'].sum()

plt.figure(figsize=(8,5))
sns.lineplot(x=monthly_sales.index, y=monthly_sales.values, marker='o')
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.show()

# **Monthly Sales Trend:**

**Insight**: There is a clear seasonal trend in sales, with a noticeable increase in total sales during the later months of the year, particularly peaking in December and September. Sales are generally lower in the middle months of the year (e.g., June). This suggests potential for holiday or end-of-year promotions.

In [ ]:
top_products = df.groupby('Product_Name')['Total_Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title("Top 10 Products by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product")
plt.show()

# **Top 10 Products by Sales:**

**Insight**: 'Headphones' is the top-selling product, significantly outperforming other items. Products like 'Jeans', 'Eggs', and 'Rice' also rank high. This highlights specific product categories that are highly popular and drive a large portion of the revenue.

In [ ]:
customer_sales = df.groupby('Customer_ID')['Total_Sales'].sum().sort_values(ascending=False).head(5)

plt.figure(figsize=(10,5))
sns.barplot(x=customer_sales.values, y=customer_sales.index)
plt.title("Top 5 Customers by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Customer ID")
plt.xticks(rotation=45) # Rotate x-axis labels to prevent overlap
plt.show()

# **Top 5 Customers by Sales:**

**Insight**: Customer ID 94 is the highest-value customer, contributing substantially more to total sales than others. Identifying and retaining these top customers can be crucial for business growth.


In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(x='Quantity', y='Total_Sales', data=df)
plt.title("Quantity vs Total Sales")
plt.show()

# **Quantity vs Total Sales:**

**Insight**: There appears to be a strong positive linear relationship between the 'Quantity' of items sold and 'Total Sales'. This is expected, as selling more items generally leads to higher total revenue, but the plot confirms this direct correlation without significant outliers suggesting unusual pricing or discounts based on quantity.

# **STEP 3: Data Preprocessing**

In [ ]:
# Check duplicates
df.duplicated().sum()

# Remove duplicates if any
df = df.drop_duplicates()

In [ ]:
# Boxplot to detect outliers
plt.figure(figsize=(8,5))
sns.boxplot(data=df[['Price', 'Total_Sales']])
plt.show()

In [ ]:
# Revenue per unit
df['Revenue_per_Unit'] = df['Total_Sales'] / df['Quantity']

# Day of Week
df['Day_of_Week'] = df['Date'].dt.day_name()

df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Category'] = le.fit_transform(df['Category'])
df['Product_Name'] = le.fit_transform(df['Product_Name'])
df['Day_of_Week'] = le.fit_transform(df['Day_of_Week'])

In [ ]:
# Normalize data for better model performance
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numerical_cols = ['Quantity', 'Price', 'Revenue_per_Unit']
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# **STEP 4: Model Building**

In [ ]:
# Features (X) and Target (y)
X = df[['Customer_ID', 'Category', 'Quantity', 'Price', 'Revenue_per_Unit', 'Month']]
y = df['Total_Sales']

In [ ]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

**Model 1: Linear Regression**

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

**Model 2: Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

**Model 3: Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(y_test, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("R2 Score:", r2_score(y_test, y_pred))
    print()

evaluate_model(y_test, y_pred_lr, "Linear Regression")
evaluate_model(y_test, y_pred_dt, "Decision Tree")
evaluate_model(y_test, y_pred_rf, "Random Forest")

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred_rf)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted (Random Forest)")
plt.show()

# **STEP 5: Hyperparameter Tuning**

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# Define model
rf = RandomForestRegressor(random_state=42)

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}

# Grid Search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2
)

# Train
grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

In [ ]:
best_rf = grid_search.best_estimator_

# Predict
y_pred_best = best_rf.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

print("Improved Model Performance:")
print("R2 Score:", r2_score(y_test, y_pred_best))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best)))

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred_best)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Tuned Model: Actual vs Predicted")
plt.show()

# **STEP 6: Final Insights + Conclusion**

In [ ]:
# Compare all models together (example structure)

print("Final Model Comparison:")

print("Linear Regression R2:", r2_score(y_test, y_pred_lr))
print("Decision Tree R2:", r2_score(y_test, y_pred_dt))
print("Random Forest R2:", r2_score(y_test, y_pred_rf))
print("Tuned Random Forest R2:", r2_score(y_test, y_pred_best))

Tuned Random Forest performed best with highest R² score, indicating strong predictive capability.

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred_best)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Final Model Performance")
plt.show()

**Business Insights**

1. Electronics category generates highest revenue.
2. Sales show monthly variation indicating seasonal trends.
3. Few customers contribute significantly to total sales.
4. Quantity and price strongly influence total sales.
5. Feature engineering improved model performance.

# Conclusion:

In this project, we analyzed retail sales data and built machine learning models to predict total sales.
After comparing multiple models, the tuned Random Forest model performed best.

The project demonstrates how data analysis and machine learning can help businesses understand sales patterns and improve decision-making.

In [ ]:
import joblib

# Save model
joblib.dump(best_rf, 'retail_sales_model.pkl')

print("Model saved successfully!")